# 开放车间调度问题 (OSSP)

**类别：** 调度

来源：[https://www.hexaly.com/templates/open-shop-scheduling-problem-ossp](https://www.hexaly.com/templates/open-shop-scheduling-problem-ossp)


## 问题描述

**在开放车间调度问题 (OSSP)** 中，一组作业必须在车间内的每台机器上进行处理。每个作业由一组无序的任务（称为活动）组成。一个活动表示该作业在一台机器上的处理过程，具有给定的处理时间。每个作业在每台机器上都有一个活动，并且当该作业的另一个活动仍在运行时，不能开始新的活动。每台机器一次只能处理一个活动。目标是找到一个使 makespan（所有作业处理完成的时间）最小的作业排序方案。

	

### 建模要点

- 添加 [区间决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/intervalvariables.html) 来建模活动
- 添加 [list 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模每个作业和每台机器上的活动顺序
- 定义 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 将区间变量和 list 变量联系起来


## 数据

我们提供的开放车间调度问题 (OSSP) 实例来自 [Taillard](http://mistic.heig-vd.ch/taillard/problemes.dir/ordonnancement.dir/ordonnancement.html)。数据文件的格式如下：

- 第一行：作业数、机器数、用于生成实例的种子、之前找到的上界和下界
- 对每个作业：每个活动在其指定机器上的处理时间
- 对每个作业：分配给每个活动的机器 ID。


## 模型

开放车间调度问题 (OSSP) 的 OptAgent 模型使用区间决策变量来表示活动的时间范围。区间的长度由每个活动的处理时间约束。

除了区间变量外，我们还使用 [list 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html)。与 [Job Shop](https://www.hexaly.com/example/job-shop-scheduling-problem-jssp) 示例类似，list 对机器上或作业内的活动进行排序。通过使用 **count** 算子约束 list 的大小，我们确保每个作业在每台机器上都被处理。

析取资源约束（即每台机器一次只能处理一个活动）可以重新表述如下：对所有 i，在位置 i+1 处理的活动必须在其位置 i 处理的活动结束后才开始。为了建模这些约束，我们将区间决策（时间范围）与 list 决策（作业排序）配对。我们编写一个 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来表达两个连续活动之间的关系。该函数用于每台机器所处理的所有活动的可变参数 **and** 算子中。

我们使用相同的策略来建模活动的析取约束。对所有作业和所有 i，作业中位置 i+1 的活动必须在该作业中位置 i 的活动结束后才开始。与析取资源约束类似，我们使用一个 lambda 函数结合可变参数 **and** 算子来建模这些约束，覆盖构成每个作业的所有活动。

目标是最小化 makespan，即所有活动处理完成的时间。


## Python 实现


In [1]:
from pathlib import Path

from optagent import OptModel, solve


def read_instance(filename):
    # The input files follow the "Taillard" format
    with open(filename, 'r') as f:
        lines = f.readlines()

    first_line = lines[1].split()
    nb_jobs = int(first_line[0])
    nb_machines = int(first_line[1])

    # Processing times for each job on each machine
    # (given in the task order, the processing order is a decision variable)
    processing_times_task_order = [[int(proc_time) for proc_time in line.split()]
                                   for line in lines[3:3 + nb_jobs]]

    # Index of machines for each task
    machine_index = [[int(machine_i) - 1 for machine_i in line.split()]
                     for line in lines[4 + nb_jobs:4 + 2 * nb_jobs]]

    # Reorder processing times: processingTime[j][m] is the processing time of the
    # task of job j that is processed on machine m
    processing_times = [[processing_times_task_order[j][machine_index[j].index(m)]
                         for m in range(nb_machines)] for j in range(nb_jobs)]

    # Trivial upper bound for the end time of tasks
    max_end = sum(map(lambda processing_times_job: sum(processing_times_job), processing_times))

    return nb_jobs, nb_machines, processing_times, max_end


def main(instance_file, output_file=None, time_limit=60):
    nb_jobs, nb_machines, processing_times, max_end = read_instance(instance_file)
    model = OptModel()

    tasks = [
        [model.interval(0, max_end, name=f"job_{job}_machine_{machine}")
         for machine in range(nb_machines)]
        for job in range(nb_jobs)
    ]
    for job in range(nb_jobs):
        for machine in range(nb_machines):
            model.constraint(
                tasks[job][machine].length() == processing_times[job][machine],
                name=f"duration_{job}_{machine}",
            )

    task_array = model.array(tasks)
    jobs_order = [
        model.list(nb_jobs, name=f"machine_{machine}_jobs")
        for machine in range(nb_machines)
    ]


    for machine, sequence in enumerate(jobs_order):
        model.constraint(sequence.count() == nb_jobs, name=f"machine_{machine}_all_jobs")
        model.constraint(
            model.and_(model.range(0, nb_jobs - 1), 
            model.lambda_function(
                lambda position: task_array[sequence[position // 1], machine]
                < task_array[sequence[(position + 1) // 1], machine]
            )),
            name=f"machine_{machine}_no_overlap",
        )

    machines_order = [
        model.list(nb_machines, name=f"job_{job}_machines")
        for job in range(nb_jobs)
    ]

    for job, sequence in enumerate(machines_order):
        model.constraint(sequence.count() == nb_machines, name=f"job_{job}_all_machines")
        model.constraint(
            model.and_(model.range(0, nb_machines - 1), 
            model.lambda_function(
                lambda position: task_array[job, sequence[position // 1]]
                < task_array[job, sequence[(position + 1) // 1]]
            )),
            name=f"job_{job}_no_overlap",
        )

    makespan = model.max(
        [tasks[job][machine].end() for job in range(nb_jobs) for machine in range(nb_machines)]
    )
    model.minimize(makespan, name="makespan")
    solution = solve(model, time_limit_s=float(time_limit))
    if not solution.feasible:
        print(f"No feasible schedule found; Status = {solution.feasible}")
        return solution

    print(f"Jobs = {nb_jobs}; Machines = {nb_machines}; Makespan = {makespan.value}; Status = {solution.feasible}")
    lines = []
    for machine, sequence in enumerate(jobs_order):
        text = " ".join(str(job + 1) for job in sequence.value)
        print(f"Machine {machine + 1}: {text}")
        lines.append(text)
    if output_file is not None:
        Path(output_file).write_text(f"{makespan.value}\n" + "\n".join(lines) + "\n", encoding="utf-8")
    return solution


## 运行实例

以下代码格演示如何调用 OptAgent 的开放车间调度模型。

In [2]:
INSTANCE_DIR = Path.cwd() / "instances"
print("Instances:", INSTANCE_DIR)


Instances: /Users/dongbox/work/opt-agent/examples/examples/hexaly/open_shop_scheduling_problem_ossp/instances


In [3]:
solution_tai44 = main(INSTANCE_DIR / "tai44_0.txt", time_limit=1)


Starting OptAgent
Parameters: time_limit=1s
[   0.003s] initial feasible=false violations=32 objective=[0]
[   0.005s] best #1 worker=0 feasible=false violations=32 objective=[671]
[   0.402s] best #24 worker=2 feasible=false violations=12 objective=[95]
[   0.677s] best #27 worker=2 feasible=false violations=9 objective=[95]
Solve summary:
  status: INFEASIBLE
  objective: [95]
  improvements: 27
  evaluated: 1530
  wall_time: 1.00743s
  termination: deadline
No feasible schedule found; Status = False
